# 🐍 Python'a Hızlı Geçiş — 2. Hafta: Kontrol Akışı + Comprehension
### Uygulama Defteri

**Bu defter bir ANLATIM aracı değildir.** Truthy/falsy'den mini proje v0.2'ye kadar tüm konuyu
öğretmeniniz az önce slaytlardan anlattı. Şimdi sıra sizde: aşağıdaki her hücreyi kendiniz yazıp
çalıştırarak az önce anlatılanı uygulayın.

**Geçen hafta:** fonksiyonlar (tip ipucu, varsayılan parametre), `mutable default argument` tuzağı,
`is None` kalıbı. `parola_istatistikleri()` fonksiyonumuz ham ölçümler üretiyordu.

**Bugün:** if/for'u Python'un kendi idiomlarıyla (truthy/falsy, ternary, comprehension) kullanıp,
`try/except` ile sağlamlaştırarak fonksiyonumuzu gerçek bir **karar motoruna** dönüştürüyoruz.

**Bugünün mini projesi:** Parola Gücü Ölçer v0.2 — artık büyük/küçük/rakam/özel karakter
**sayıyor** ve zayıf/orta/güçlü **kararı** veriyor.

---
⏱️ *Bu defter, bugünün tam akışının (180 dk) "Jupyter Uygulama" bölümüdür: ön-okuma kontrolü
(15 dk, sözlü) → Slayt Anlatımı (70 dk) → ara (10 dk) → **bu defter + Lab Alıştırmaları (75 dk)**
→ kapanış (10 dk). Bölüm pratiğine ~15 dk, Lab Alıştırmaları'na (7 görev) ~60 dk ayırın. Lab
Alıştırmaları'nın görev metinleri bu defterde DEĞİL, Öğrenci Lab Rehberi'ndedir.*


## 0. Isınma — Truthy / Falsy

Java/C'de bir `if` koşuluna yalnızca `bool` konabilir. Python'da **her değer** bir koşuldur.
Slaytta anlatılan şaşırtıcı örneği şimdi siz çalıştırın:

In [1]:
bos_liste = []
if bos_liste:
    print("Liste dolu")
else:
    print("Liste boş")

Liste boş


Hata yok — ama Java/C alışkanlığıyla `if bos_liste == True:` gibi bir şey beklemiş olabilirsiniz.
Python'da boş bir koleksiyon **otomatik olarak** falsy'dir. Devam edelim:

In [2]:
print(bool(0), bool(""), bool([]), bool(None))   # hepsi False
print(bool(1), bool("a"), bool([0]), bool(" "))  # hepsi True - [0] bile!

False False False False
True True True True


🤔 **Tartışalım:** `bool([0])` neden `True`? (İpucu: liste **boş değil** — içinde bir eleman var,
o elemanın değerinin `0` olması önemli değil. Falsy/truthy, elemanın değerine değil, koleksiyonun
**boş olup olmadığına** bakar.)

## 1. Ternary İfade ve `is` vs `==`

Java/C'deki `? :` operatörünün Python karşılığı kelimelerle yazılır:

In [3]:
yas = 20
sonuc = "yetişkin" if yas >= 18 else "çocuk"
print(sonuc)

yetişkin


Şimdi `is` ile `==` arasındaki farkı siz test edin. Önce beklediğiniz gibi çalışan bir örnek:

In [4]:
a = [1, 2, 3]
b = [1, 2, 3]
print(a == b)   # True - degerce ayni
print(a is b)   # False - iki FARKLI liste nesnesi, sadece icerikleri ayni

True
False


### 🐍 Python Sürprizi — Küçük Tam Sayı Önbellekleme (Interning)

Slaytta anlatılan, deneyimli programcıları bile şaşırtan bu davranışı şimdi siz üretin:

In [5]:
a = 100
b = 100
print(a is b)   # ne bekliyorsunuz?

c = 300
d = 300
print(c is d)   # ayni mantikla ayni sonucu mu verir?

True
True


**İkisi de farklı davranabilir** (ilk `True`, ikinci ortama göre `False` olabilir)! Sebep:
CPython, performans için **-5 ile 256 arasındaki** küçük tam sayıları önceden oluşturup
**yeniden kullanır** (interning) — bu bir dil kuralı değil, CPython'a özgü bir **uygulama
detayı**dır ve resmi olarak garanti edilmez.

👉 **Çıkarım (ezberlenecek kural):** Sayıları, string'leri ya da herhangi bir **değeri**
karşılaştırırken her zaman `==` kullanın. `is`'i yalnızca `None`, `True`, `False` gibi
**tekil (singleton) nesneler** için kullanın. Bu davranışa güvenerek kod yazmak, farklı
Python sürümünde ya da farklı makinede sessizce bozulabilecek bir koddur.

## 2. Döngüler: for, range, enumerate, zip

Python'un `for`'u Java'nın for-each'ine benzer:

In [6]:
parola = "Ab12!"
for karakter in parola:
    print(karakter)

A
b
1
2
!


`enumerate()` ile hem indeks hem değeri birlikte alırız — Java'daki klasik sayaçlı döngüye gerek kalmaz:

In [7]:
for i, karakter in enumerate(parola):
    print(f"{i}: {karakter}")

0: A
1: b
2: 1
3: 2
4: !


### 🐍 Python Sürprizi — for...else

Java/C'de karşılığı olmayan bir özellik: döngünün `else` bloğu, yalnızca döngü **break ile
kesilmeden** bittiğinde çalışır:

In [8]:
for karakter in parola:
    if karakter == " ":
        print("Boşluk bulundu!")
        break
else:
    print("Hiç boşluk yok, tüm parola tarandı")

Hiç boşluk yok, tüm parola tarandı


👉 **Çıkarım:** `for...else`, "ara ve bulamazsan bir şey yap" senaryosunda, ayrı bir
`bulundu = True/False` bayrağı tutmaktan daha temiz bir çözümdür.

## 3. Liste Comprehension — Geçen Haftanın Köprüsünü Kapatıyoruz

Hatırlayın: geçen hafta *"parolada kaç tane BÜYÜK harf var?"* sorusunu **cevaplayamamıştık**.
Şimdi cevaplayabiliriz:

In [9]:
buyukler = [k for k in parola if k.isupper()]
print(buyukler)
print(len(buyukler))

['A']
1


Genel kalıp: **[ifade for eleman in iterable if koşul]**. Aynı mantık dict comprehension
için de geçerlidir:

In [10]:
sayac = {
    "buyuk_harf": len([k for k in parola if k.isupper()]),
    "kucuk_harf": len([k for k in parola if k.islower()]),
    "rakam": len([k for k in parola if k.isdigit()]),
}
print(sayac)

{'buyuk_harf': 1, 'kucuk_harf': 1, 'rakam': 2}


## 4. Hata Yönetimi: try / except — Geçen Haftanın Kasıtlı Hatasını Çözüyoruz

Hatırlayın: geçen hafta `int(input(...))` çağrısına kullanıcı bir kelime yazarsa program
**çöküyordu** (`ValueError`). Şimdi bunu yakalayalım:

In [11]:
try:
    deneme_yas = int(input("Bir sayı girin: "))
    print(f"Girdiğiniz sayı: {deneme_yas}")
except ValueError:
    print("Bu bir sayı değildi! Varsayılan olarak 0 kullanılıyor.")
    deneme_yas = 0

Bir sayı girin: 42
Girdiğiniz sayı: 42


👉 **Çıkarım:** `try/except`, programı **çökertmeden** hatayı ele almanın yoludur — dönem
projesindeki "savunmacı programlama" gereksiniminin temel aracı budur.

## 5. 🎯 Mini Proje: Parola Gücü Ölçer v0.2 — Artık Gerçek Bir Karar Motoru

Geçen haftaki `parola_istatistikleri()` fonksiyonunu genişletelim: comprehension ile detaylı
sayım yapan, ve ayrı bir fonksiyonla "zayıf/orta/güçlü" **kararı** veren iki fonksiyon
yazacağız.

In [12]:
def parola_istatistikleri(parola: str) -> dict:
    """Bir parolanin detayli istatistiklerini hesaplar (comprehension ile)."""
    return {
        "uzunluk": len(parola),
        "buyuk_harf": len([k for k in parola if k.isupper()]),
        "kucuk_harf": len([k for k in parola if k.islower()]),
        "rakam": len([k for k in parola if k.isdigit()]),
        "ozel_karakter": len([k for k in parola if not k.isalnum()]),
    }

Şimdi bu istatistiklere bakıp bir **karar** veren ikinci fonksiyonu yazalım:

In [13]:
def parola_gucunu_degerlendir(istatistik: dict) -> str:
    """Istatistiklere gore zayif/orta/guclu karari verir."""
    puan = 0
    if istatistik["uzunluk"] >= 8:
        puan += 1
    if istatistik["buyuk_harf"] > 0 and istatistik["kucuk_harf"] > 0:
        puan += 1
    if istatistik["rakam"] > 0:
        puan += 1
    if istatistik["ozel_karakter"] > 0:
        puan += 1

    if puan <= 1:
        return "zayıf"
    elif puan <= 3:
        return "orta"
    else:
        return "güçlü"

Şimdi `try/except` ile sağlamlaştırılmış bir girdi alıp, iki fonksiyonu zincirleyelim:

In [14]:
try:
    parola = input("Bir deneme parolası girin: ")
    if not parola:
        raise ValueError("Parola boş olamaz")
except ValueError as e:
    print(f"Hata: {e}")
    parola = "varsayilan123"

istatistik = parola_istatistikleri(parola)
karar = parola_gucunu_degerlendir(istatistik)

print("=" * 30)
print("PAROLA GÜÇ RAPORU")
print("=" * 30)
for anahtar, deger in istatistik.items():
    print(f"{anahtar}: {deger}")
print(f"KARAR: {karar.upper()}")

Bir deneme parolası girin: Abc123!x
PAROLA GÜÇ RAPORU
uzunluk: 8
buyuk_harf: 1
kucuk_harf: 3
rakam: 3
ozel_karakter: 1
KARAR: GÜÇLÜ


### 🚧 Bugünün Sınırı — Haftaya Köprü

İki fonksiyonumuz da çalışıyor, ama birbirinden tamamen bağımsız iki fonksiyon + bir çağrı
bloğu olarak duruyor. Gerçek bir yazılım projesinde bunları nasıl bir arada, tekrar
kullanılabilir şekilde organize ederiz? Haftaya **sınıflar (class)** ile bu soruyu
cevaplayacağız — `ParolaOlcer` adında bir nesne, hem istatistikleri hem kararı kendi içinde
tutacak.

---
## ☕ Ara — 10 dakika

Su için, biraz hareket edin, ekrandan uzaklaşın. Ara bitince eşinizi bulun — bundan sonraki
bölüm ikili çalışmadır.

---
## 🤝 Lab Alıştırmaları (60 dakika, 7 Görev)

**Kurallar:** Isınma görevi bireysel, geri kalan 6 görev ikili (tek klavye, süre bitince yer
değiştirin). Görevlerin tam metni (zorluk seviyesi, ipucu, örnek girdi/çıktı) bu defterde DEĞİL,
**Öğrenci Lab Rehberi**'ndedir — burada sadece kod hücreleri var. Takıldığınız yerde önce eşinizle
tartışın, sonra öğretmene sorun. 7 görevin hepsini bitirmek şart değil.

### Görev 1 — Isınma: Falsy Kontrolü (5 dk, bireysel)
`bos_mu(deger) -> bool` yazın: `not deger` kullanarak Python'a göre falsy olup olmadığını
döndürsün. (Lab Rehberi, Görev 1)

In [15]:
# Görev 1 - Falsy Kontrolü
def bos_mu(deger) -> bool:
    """Deger Python'a gore falsy ise True dondurur."""
    return not deger

print(bos_mu([]), bos_mu("merhaba"), bos_mu(0), bos_mu([0]), bos_mu(" "))

True False True False False


### Görev 2 — Rehberli: Yeni Bir Kriter Ekleyin (10 dk, ikili)
`parola_istatistikleri`'ne `"bosluk_var_mi"` anahtarını ekleyin (bool). `parola_gucunu_degerlendir`
fonksiyonunu, boşluk varsa puan DÜŞÜRECEK şekilde güncelleyin. (Lab Rehberi, Görev 2)

In [16]:
# Görev 2 - Yeni Bir Kriter: boşluk varsa puan düşer
def parola_istatistikleri(parola: str) -> dict:
    """Bir parolanin detayli istatistiklerini hesaplar (bosluk kriteri eklendi)."""
    return {
        "uzunluk": len(parola),
        "buyuk_harf": len([k for k in parola if k.isupper()]),
        "kucuk_harf": len([k for k in parola if k.islower()]),
        "rakam": len([k for k in parola if k.isdigit()]),
        "ozel_karakter": len([k for k in parola if not k.isalnum()]),
        "bosluk_var_mi": " " in parola,
    }


def parola_gucunu_degerlendir(istatistik: dict) -> str:
    """Istatistiklere gore zayif/orta/guclu karari verir; bosluk puan dusurur."""
    puan = 0
    if istatistik["uzunluk"] >= 8:
        puan += 1
    if istatistik["buyuk_harf"] > 0 and istatistik["kucuk_harf"] > 0:
        puan += 1
    if istatistik["rakam"] > 0:
        puan += 1
    if istatistik["ozel_karakter"] > 0:
        puan += 1
    if istatistik.get("bosluk_var_mi"):
        puan = max(puan - 1, 0)  # puan negatif olmasin

    if puan <= 1:
        return "zayıf"
    elif puan <= 3:
        return "orta"
    else:
        return "güçlü"

print(parola_gucunu_degerlendir(parola_istatistikleri("Abc123!x")))
print(parola_gucunu_degerlendir(parola_istatistikleri("Abc 123!x")))

güçlü
orta


### Görev 3 — Rehberli: Comprehension ile Liste Üretin (10 dk, ikili)
Kullanıcıdan virgülle ayrılmış birkaç parola alın, her biri için `parola_gucunu_degerlendir`
çağırıp sonuçları TEK bir liste comprehension ile toplayın. (Lab Rehberi, Görev 3)

In [17]:
# Görev 3 - Comprehension ile Liste Üretin
parolalar = input("Parolalari virgulle ayirarak girin: ").split(",")
sonuclar = [parola_gucunu_degerlendir(parola_istatistikleri(p.strip())) for p in parolalar]
print(sonuclar)

Parolalari virgulle ayirarak girin: abc, Abc12345 ,Abc123!x
['zayıf', 'orta', 'güçlü']


### Görev 4 — Bağımsız: for...else ile Arama (10 dk, ikili)
Bir parola listesinde "güçlü" olarak değerlendirilen İLK parolayı bulan bir döngü yazın;
for...else kullanarak hiç güçlü parola yoksa özel bir mesaj yazdırın. (Lab Rehberi, Görev 4)

In [18]:
# Görev 4 - for...else ile Arama
parola_listesi = ["abc", "Abc12345", "Abc123!x", "Zz9!zzzz"]
for p in parola_listesi:
    if parola_gucunu_degerlendir(parola_istatistikleri(p)) == "güçlü":
        print(f"Ilk guclu parola: {p}")
        break
else:
    print("Listede hic guclu parola yok!")

for p in ["abc", "12345"]:
    if parola_gucunu_degerlendir(parola_istatistikleri(p)) == "güçlü":
        print(f"Ilk guclu parola: {p}")
        break
else:
    print("Listede hic guclu parola yok!")

Ilk guclu parola: Abc123!x
Listede hic guclu parola yok!


### Görev 5 — Bağımsız: Geçen Haftaki Göreve Köprü (10 dk, ikili)
Geçen hafta yazdığınız `parola_tekrar_sayisi()` fonksiyonunu, bugün öğrendiğiniz dict comprehension
kullanarak TEK satırda yeniden yazın. (Lab Rehberi, Görev 5)

In [19]:
# Görev 5 - parola_tekrar_sayisi (dict comprehension ile)
def parola_tekrar_sayisi(parola: str) -> dict:
    """En cok tekrar eden karakteri ve adedini dondurur."""
    sayac = {k: parola.count(k) for k in parola}
    en_cok = max(sayac, key=sayac.get)
    return {"karakter": en_cok, "adet": sayac[en_cok]}

print(parola_tekrar_sayisi("mississippi"))

{'karakter': 'i', 'adet': 4}


### Görev 6 — Hata Ayıklama: Program Çalışmıyor (7 dk, ikili)
Aşağıdaki hücreyi çalıştırın, hatayı okuyun, düzeltin. Hangi hata türü olduğunu açıklayın.
(Lab Rehberi, Görev 6)

In [20]:
# Görev 6 - DÜZELTİLDİ
# Hata: if satirinin sonunda ":" eksikti -> SyntaxError (expected ':')
def parola_gucunu_degerlendir_v2(istatistik: dict) -> str:
    """Uzunluga gore basit zayif/guclu karari verir."""
    puan = 0
    if istatistik["uzunluk"] >= 8:
        puan += 1
    return "zayif" if puan == 0 else "guclu"

print(parola_gucunu_degerlendir_v2({"uzunluk": 3}))
print(parola_gucunu_degerlendir_v2({"uzunluk": 10}))

zayif
guclu


### Görev 7 — Hata Ayıklama: Yanlış Sonuç Veriyor (8 dk, ikili)
Bu hücre ÇALIŞIYOR ama yanlış sonuç veriyor — hata mesajı yok (sessiz mantık hatası). Bulun ve
düzeltin. (Lab Rehberi, Görev 7)

In [21]:
# Görev 7 - DÜZELTİLDİ
# Hata: if istatistik["uzunluk"]: -> 0 disindaki her sayi truthy oldugu icin 3 de True veriyordu.
# Hata mesaji yok cunku Python'da her deger kosul olabilir -> sessiz mantik hatasi.
def guclu_mu(istatistik: dict) -> bool:
    """Parolanin en az 8 karakter olup olmadigini kontrol eder."""
    return istatistik["uzunluk"] >= 8

print(guclu_mu({"uzunluk": 3}))   # False
print(guclu_mu({"uzunluk": 12}))  # True

False
True


> 🔎 Öğretmeniniz bu süre boyunca sınıfta dolaşacak ama doğrudan cevap vermeyecek —
> takıldığınız yerde önce siz ve eşiniz tartışın, sonra sorun.

---
## ✅ Ders Sonrası (Bugün Yapılacaklar)

1. **GitHub'a commit:** Bu defteri (ve Lab Alıştırmaları çözümlerinizi) reponuza yükleyin.
2. **README güncelleyin:** Bugün ne öğrendiğinizi 2-3 cümleyle ekleyin.
3. **Haftaya hazırlık:** Haftaya konumuz **sınıflar (class)** — bugünkü iki bağımsız
   fonksiyonumuzu tek bir `ParolaOlcer` nesnesinde birleştireceğiz.

---
### 📌 Bugünün Tek Cümlesi
> *Python size "truthy/falsy", "for...else", "comprehension" gibi kısayollar sunar — ama
> `is` ile `==`'yi karıştırmak gibi kısayolların bedelini de unutmayın.*
